In [ ]:
!pip install -q tifffile imagecodecs

import os, random, time
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms.functional as F

In [ ]:
class HistoricBuildingDataset(Dataset):
    def __init__(self, root, class_ids=(1,2,3), image_ext=".tif"):
        self.root = root
        self.images_dir = os.path.join(root, "images")
        self.labels_dir = os.path.join(root, "labels")
        self.class_ids = tuple(class_ids)
        self.image_paths = sorted([
            os.path.join(self.images_dir, f)
            for f in os.listdir(self.images_dir)
            if f.lower().endswith(image_ext)
        ])

        # Check if the file has a matching class id, drop otherwhise
        valid = []
        for p in self.image_paths:
            name = os.path.basename(p)
            has_any = False
            for cid in self.class_ids:
                mp = os.path.join(self.labels_dir, str(cid), name)
                if os.path.exists(mp):
                    has_any = True
                    break # If even one label file exists, keep the image

            if has_any:
                valid.append(p)
        self.image_paths = valid

    def __len__(self):
        return len(self.image_paths)

    def _read_image(self, path):
        img = tifffile.imread(path)
        if img.ndim == 2:
            img = np.stack([img]*3, axis=-1)
        elif img.ndim == 3 and img.shape[-1] == 1:
            img = np.concatenate([img]*3, axis=-1)
        if np.issubdtype(img.dtype, np.floating):
            img = (img * 255).astype(np.uint8)
        else:
            img = img.astype(np.uint8)
        return img

    def _read_mask(self, path):
        try:
            m = tifffile.imread(path)
            if m.ndim == 3:
                m = m.max(axis=-1)
            return m
        except Exception:
            return None

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = self._read_image(img_path)
        H, W = img.shape[:2]

        boxes = []
        labels = []
        instance_masks = []

        base = os.path.basename(img_path)
        for class_id in self.class_ids:
            mask_path = os.path.join(self.labels_dir, str(class_id), base)
            if not os.path.exists(mask_path):
                continue
            mask_raster = self._read_mask(mask_path)
            if mask_raster is None:
                continue

            unique_ids = np.unique(mask_raster)
            unique_ids = unique_ids[unique_ids != 0]
            for uid in unique_ids:
                single = (mask_raster == uid).astype(np.uint8)
                if single.sum() == 0:
                    continue
                ys, xs = np.where(single)
                ymin, ymax = int(ys.min()), int(ys.max())
                xmin, xmax = int(xs.min()), int(xs.max())
                if xmax <= xmin or ymax <= ymin:
                    continue
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(int(class_id))
                if single.shape != (H, W):
                    canvas = np.zeros((H, W), dtype=np.uint8)
                    h0, w0 = single.shape
                    canvas[:h0, :w0] = single
                    instance_masks.append(canvas)
                else:
                    instance_masks.append(single)

        if len(boxes) == 0:
            boxes = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            masks = torch.zeros((0, H, W), dtype=torch.uint8)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            masks = torch.as_tensor(np.stack(instance_masks, axis=0), dtype=torch.uint8)

        image = F.to_tensor(img)  # [C,H,W] float32
        target = {"boxes": boxes, "labels": labels, "masks": masks}
        return image, target


In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Initialize the Dataset
data_path = '/content/drive/MyDrive/Deep_learning_test'
dataset = HistoricBuildingDataset(data_path)

# Split into Training and Validation sets
torch.manual_seed(1)
indices = torch.randperm(len(dataset)).tolist()
split_idx = int(len(dataset) * 0.9)

dataset_train = torch.utils.data.Subset(dataset, indices[:split_idx])
dataset_test = torch.utils.data.Subset(dataset, indices[split_idx:])

# Colab GPU can handle a batch size of 4
data_loader = DataLoader(dataset_train, batch_size=4, shuffle=True, collate_fn=collate_fn)

print(f"Loaded {len(dataset_train)} training chips and {len(dataset_test)} validation chips.")

In [ ]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def get_model(num_classes):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

    # Replace bounding box head
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Replace mask head
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model

# Send model to the Colab GPU
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = get_model(num_classes=4)
model.to(device)

# Setup the optimizer (how the model learns from mistakes)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

print(f"Model built and loaded onto: {device}")

In [ ]:
#training the model
num_epochs = 10
print("Starting training loop...")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    start_time = time.time()

    for images, targets in data_loader:
        # Move data to GPU
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Calculate loss
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        epoch_loss += losses.item()

        # Backpropagation (updating the brain)
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{num_epochs} | Average Loss: {epoch_loss/len(data_loader):.4f} | Time: {epoch_time:.0f}s")

# Save the trained brain to your Drive
torch.save(model.state_dict(), '/content/drive/MyDrive/Deep_learning_model/historic_building_model.pth')
print("Training complete and model successfully saved to Drive!")

Output of previous cell:

Starting training loop...

Epoch 1/10 | Average Loss: 0.5496 | Time: 3856s

Epoch 2/10 | Average Loss: 0.4768 | Time: 1497s

Epoch 3/10 | Average Loss: 0.4501 | Time: 1503s

Epoch 4/10 | Average Loss: 0.4216 | Time: 1503s

Epoch 5/10 | Average Loss: 0.4040 | Time: 1505s

Epoch 6/10 | Average Loss: 0.3862 | Time: 1509s

Epoch 7/10 | Average Loss: 0.3699 | Time: 1512s

Epoch 8/10 | Average Loss: 0.3558 | Time: 1511s

Epoch 9/10 | Average Loss: 0.3423 | Time: 1513s

Epoch 10/10 | Average Loss: 0.3307 | Time: 1516s

Training complete and model successfully saved to Drive!

In [ ]:
import os
import torch
import torchvision
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


from google.colab import drive
drive.mount('/content/drive')

# 1. Define paths
MODEL_PATH = '/content/drive/MyDrive/Pei_building_model/historic_building_model.pth'

# UPDATE THIS: Change to wherever you uploaded your folder of screenshots!
FOLDER_PATH = '/content/drive/MyDrive/princecounty_images'

#confidence level
CONFIDENCE_THRESHOLD = 0.50
CLASS_MAP = {1: "Barn", 2: "House", 3: "Other"}

def get_trained_model(num_classes=4):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    return model

# Initialize CPU engine
device = torch.device('cpu')
print("Loading model weights onto CPU...")
model = get_trained_model(num_classes=4)
checkpoint = torch.load(MODEL_PATH, map_location=device)
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)
model.eval()
print("Model loaded and standing by!")

# 2. Loop through the test folder
if not os.path.exists(FOLDER_PATH):
    print(f"[ERROR] Could not find the folder at {FOLDER_PATH}. Please double check the path.")
else:
    # Grab all valid image files from the directory
    valid_extensions = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
    image_files = [f for f in os.listdir(FOLDER_PATH) if f.lower().endswith(valid_extensions)]

    print(f"Found {len(image_files)} sample files to process.\n")

    for filename in image_files:
        full_image_path = os.path.join(FOLDER_PATH, filename)
        print(f"Processing: {filename}...")

        # Load and convert image to numpy array layout
        pil_img = Image.open(full_image_path).convert("RGB")
        img_uint8 = np.array(pil_img)

        # Build PyTorch input tensor [C, H, W] scaled to [0, 1]
        input_tensor = torch.as_tensor(img_uint8, dtype=torch.float32).permute(2, 0, 1) / 255.0
        input_tensor = input_tensor.unsqueeze(0).to(device)

        # Run detection
        with torch.no_grad():
            predictions = model(input_tensor)[0]

        boxes = predictions['boxes'].cpu().numpy()
        labels = predictions['labels'].cpu().numpy()
        scores = predictions['scores'].cpu().numpy()
        masks = predictions['masks'].cpu().squeeze(1).cpu().numpy()

        # Render setup
        fig, ax = plt.subplots(1, 1, figsize=(5, 5))
        ax.imshow(img_uint8)

        detection_count = 0
        for i in range(len(scores)):
            if scores[i] < CONFIDENCE_THRESHOLD:
                continue

            detection_count += 1
            xmin, ymin, xmax, ymax = boxes[i]
            label_id = labels[i]

            # Draw cyan bounding box
            rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, linewidth=2, edgecolor='cyan', facecolor='none')
            ax.add_patch(rect)

            # Overlay translucent prediction mask
            mask = masks[i] > 0.5
            mask_overlay = np.zeros((*mask.shape, 4))
            mask_overlay[mask] = [0.0, 1.0, 1.0, 0.3] # Cyan mask at 30% alpha
            ax.imshow(mask_overlay)

            label_name = CLASS_MAP.get(label_id, f"Class {label_id}")
            ax.text(xmin, ymin - 4, f"{label_name} ({scores[i]*100:.1f}%)",
                    color='cyan', fontsize=9, bbox=dict(facecolor='black', alpha=0.6, pad=1))

        plt.title(f"File: {filename} | Detections: {detection_count}", fontsize=14)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        print("-" * 50)

Example of model output based on PEI map 1968:

### Mask R-CNN Detection Results

![Detection 1](images/Screenshot%202026-07-17%20104807.png)

![Detection 2](images/Screenshot%202026-07-17%20104926.png)

![Detection 3](images/Screenshot%202026-07-17%20105017.png)

![Detection 4](images/Screenshot%202026-07-17%20105041.png)

![Detection 5](images/Screenshot%202026-07-17%20105107.png)

![Detection 6](images/Screenshot%202026-07-17%20105518.png)

![Detection 7](images/Screenshot%202026-07-17%20110358.png)
